# Data Preparation Lab -- Module 2, Class 2

**Dataset:** Superstore Sales

In this lab you will:
1. Load and inspect data
2. Handle missing values
3. Remove duplicates
4. Convert data types
5. Create derived features

Tasks 1-3 are provided as guided starter steps. Tasks 4-6 are completed below with beginner-friendly code and notes.


---
## Setup: Load the Dataset

The assignment dataset is the Superstore Sales CSV. This notebook first looks for a CSV file that you uploaded, then tries public Superstore CSV links.

- If you are using Google Colab, upload `SampleSuperstore.csv` if needed.
- If a public link works, the notebook can run without manual upload.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1) Try common local file names first. This is useful when you upload the CSV to Colab.
possible_files = [
    'SampleSuperstore.csv',
    'Sample - Superstore.csv',
    'Superstore.csv',
    'superstore.csv',
]

df = None
for file_name in possible_files:
    if Path(file_name).exists():
        df = pd.read_csv(file_name, encoding='latin-1')
        print(f"Loaded local file: {file_name}")
        break

# 2) If no local file is available, try public URLs.
# The first URL was in the starter notebook. The second URL is a working raw CSV fallback.
if df is None:
    urls = [
        'https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv',
        'https://gist.githubusercontent.com/nnbphuong/38db511db14542f3ba9ef16e69d3814c/raw/3a77ff9d97c504d3ec3210b12fde7242b8c6ab63/Superstore.csv',
    ]

    last_error = None
    for url in urls:
        try:
            df = pd.read_csv(url, encoding='latin-1')
            print(f"Loaded from URL: {url}")
            break
        except Exception as error:
            last_error = error
            print(f"Could not load this URL: {url}")

# 3) Final fallback for Google Colab: ask the user to upload the CSV.
if df is None:
    try:
        from google.colab import files

        print('Please upload SampleSuperstore.csv now.')
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        df = pd.read_csv(filename, encoding='latin-1')
        print(f"Loaded from upload: {filename}")
    except Exception as error:
        raise RuntimeError(
            'Could not load the Superstore dataset. Upload SampleSuperstore.csv and run this cell again.'
        ) from last_error

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")


Could not load this URL: https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv


Loaded from URL: https://gist.githubusercontent.com/nnbphuong/38db511db14542f3ba9ef16e69d3814c/raw/3a77ff9d97c504d3ec3210b12fde7242b8c6ab63/Superstore.csv
Dataset loaded: 9994 rows, 21 columns


---
## Task 1: Inspect the Data (pre-built)

Always look at your data before doing anything to it.


In [2]:
# First 5 rows
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
# Shape: rows x columns
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Shape: 9994 rows, 21 columns


In [4]:
# Data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9983 non-null   float64
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

In [5]:
# Summary statistics for numerical columns
df.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9983.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55245.233297,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32038.715955,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,57103.000000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


### Task 1 Notes

The Superstore dataset has 9,994 rows and 21 columns when loaded from the provided public CSV. Most columns are text categories such as `Ship Mode`, `Segment`, `City`, and `Category`; the main numeric columns are `Sales`, `Quantity`, `Discount`, and `Profit`.

The first inspection shows that the dataset is already mostly clean. The main issue is a small number of missing `Postal Code` values.


---
## Task 2: Missing Values (pre-built)

Check which columns have missing values and how many.


In [6]:
# Count missing values per column.
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct,
})

# Keep this copy so we can explain what was filled later.
missing_report_before_fill = missing_report.copy()

# Show only columns with missing values.
missing_report_before_fill[missing_report_before_fill['missing_count'] > 0]


,missing_count,missing_pct
Postal Code,11,0.11


In [7]:
# Fill missing numerical values with the median.
# Median is a good beginner default because it is less affected by very large or very small outliers.
numerical_cols = df.select_dtypes(include=[np.number]).columns
filled_values = {}

for col in numerical_cols:
    missing_count = df[col].isnull().sum()

    if missing_count > 0:
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)

        filled_values[col] = {
            'missing_count': int(missing_count),
            'strategy': 'median',
            'value_used': median_value,
        }

# Fill missing categorical values with the mode.
# Mode means the most common value in the column.
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    missing_count = df[col].isnull().sum()

    if missing_count > 0:
        mode_value = df[col].mode()[0]
        df[col] = df[col].fillna(mode_value)

        filled_values[col] = {
            'missing_count': int(missing_count),
            'strategy': 'mode',
            'value_used': mode_value,
        }

if filled_values:
    print('Filled missing values:')
    for col, details in filled_values.items():
        print(
            f"- {col}: filled {details['missing_count']} values "
            f"using {details['strategy']} = {details['value_used']}"
        )
else:
    print('No missing values needed to be filled.')

print()
print(f"Total missing values remaining: {df.isnull().sum().sum()}")


Filled missing values:
- Postal Code: filled 11 values using median = 57103.0

Total missing values remaining: 0


C:\Temp\ipykernel_9920\2645865218.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


### Task 2 Notes

The dataset has 11 missing values in `Postal Code`. Because `Postal Code` is read as a numeric column, those missing values were filled with the column median.

No categorical columns needed filling in this dataset. After the fill step, the total number of missing values is 0.


---
## Task 3: Remove Duplicates (pre-built)


In [8]:
# Check for duplicates.
rows_before_duplicates = df.shape[0]
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {n_duplicates}")

# Remove duplicates only if they exist.
if n_duplicates > 0:
    df = df.drop_duplicates()

rows_after_duplicates = df.shape[0]
print(f"Rows before duplicate check: {rows_before_duplicates}")
print(f"Rows after duplicate check: {rows_after_duplicates}")


Duplicate rows found: 0
Rows before duplicate check: 9994
Rows after duplicate check: 9994


### Task 3 Notes

There are 0 fully duplicated rows in this dataset, so no rows were removed. A fully duplicated row means every column value is exactly the same as another row.


---
## Task 4: Convert Date Columns

The `Order Date` and `Ship Date` columns are stored as strings. Convert them to proper datetime objects.

Hint: Use `pd.to_datetime()`. After conversion, verify with `.dtypes`.


In [9]:
# Check current types of date columns
print("Before conversion:")
for col in df.columns:
    if 'date' in col.lower() or 'Date' in col:
        print(f"  {col}: {df[col].dtype}")
        print(f"  Sample value: {df[col].iloc[0]}")

Before conversion:
  Order Date: str
  Sample value: 2017-11-08
  Ship Date: str
  Sample value: 2017-11-11


In [10]:
# Convert the two date columns from text into datetime values.
date_cols = ['Order Date', 'Ship Date']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print('Date conversion complete.')


Date conversion complete.


In [11]:
# Verify the conversion worked.
print('After conversion:')
print(df[date_cols].dtypes)

print()
print('Missing or invalid date values after conversion:')
print(df[date_cols].isnull().sum())

# Spot-check the first few converted dates.
df[date_cols].head()


After conversion:
Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

Missing or invalid date values after conversion:
Order Date    0
Ship Date     0
dtype: int64


,Order Date,Ship Date
0,2017-11-08,2017-11-11
1,2017-11-08,2017-11-11
2,2017-06-12,2017-06-16
3,2016-10-11,2016-10-18
4,2016-10-11,2016-10-18


### Task 4 Notes

`Order Date` and `Ship Date` were converted from plain text into Pandas datetime columns. This matters because datetime columns can be sorted, filtered, grouped by month or year, and used for time-based calculations.

The verification cell checks both the new data types and whether any invalid dates became missing values during conversion.


---
## Task 5: Derived Features

Create customer-level summary features. These are the building blocks for customer segmentation (Activity 4).

You need to create:
- **total_spending**: Total sales per customer
- **order_frequency**: Number of orders per customer
- **avg_order_value**: Average sales amount per order per customer

Hint: Use `df.groupby('Customer ID')` (or whatever the customer ID column is named).


In [12]:
# First, identify the right column names
print("All columns:")
for col in df.columns:
    print(f"  {col}")

All columns:
  Row ID
  Order ID
  Order Date
  Ship Date
  Ship Mode
  Customer ID
  Customer Name
  Segment
  Country
  City
  State
  Postal Code
  Region
  Product ID
  Category
  Sub-Category
  Product Name
  Sales
  Quantity
  Discount
  Profit


In [13]:
# Create total_spending per customer.
# This adds together all Sales values for each Customer ID.
customer_id_col = 'Customer ID'
sales_col = 'Sales'
order_id_col = 'Order ID'

customer_spending = df.groupby(customer_id_col)[sales_col].sum()
customer_spending.name = 'total_spending'

customer_spending.head()


Customer ID
AA-10315    5563.560
AA-10375    1056.390
AA-10480    1790.512
AA-10645    5086.935
AB-10015     886.156
Name: total_spending, dtype: float64

In [14]:
# Create order_frequency per customer.
# We count unique Order ID values because one order can have multiple product rows.
order_freq = df.groupby(customer_id_col)[order_id_col].nunique()
order_freq.name = 'order_frequency'

order_freq.head()


Customer ID
AA-10315    5
AA-10375    9
AA-10480    4
AA-10645    6
AB-10015    3
Name: order_frequency, dtype: int64

In [15]:
# Create avg_order_value per customer.
# This follows the assignment hint: average Sales value for each customer.
avg_order = df.groupby(customer_id_col)[sales_col].mean()
avg_order.name = 'avg_order_value'

avg_order.head()


Customer ID
AA-10315    505.778182
AA-10375     70.426000
AA-10480    149.209333
AA-10645    282.607500
AB-10015    147.692667
Name: avg_order_value, dtype: float64

In [16]:
# Combine the three customer-level Series into one DataFrame.
customer_summary = pd.concat(
    [customer_spending, order_freq, avg_order],
    axis=1,
).reset_index()

# Sort so the highest-spending customers are easy to see first.
customer_summary = customer_summary.sort_values('total_spending', ascending=False)

print(f"Customer summary: {customer_summary.shape[0]} customers, {customer_summary.shape[1]} columns")
customer_summary.head(10)


Customer summary: 793 customers, 4 columns


,Customer ID,total_spending,order_frequency,avg_order_value
700,SM-20320,25043.050,5,1669.536667
741,TC-20980,19052.218,5,1587.684833
621,RB-19360,15117.339,6,839.852167
730,TA-21385,14595.620,4,1459.562000
6,AB-10105,14473.571,10,723.678550
434,KL-16645,14175.229,12,488.801000
669,SC-20095,14142.334,9,642.833364
327,HL-15040,12873.298,6,1170.299818
683,SE-20110,12209.438,11,642.602000
131,CC-12370,12129.072,5,1102.642909


In [17]:
# Display the first 10 rows and summary statistics.
print('First 10 customer summary rows:')
print(customer_summary.head(10).to_string(index=False))

print()
print('Summary statistics:')
customer_summary.describe()


First 10 customer summary rows:
Customer ID  total_spending  order_frequency  avg_order_value
   SM-20320       25043.050                5      1669.536667
   TC-20980       19052.218                5      1587.684833
   RB-19360       15117.339                6       839.852167
   TA-21385       14595.620                4      1459.562000
   AB-10105       14473.571               10       723.678550
   KL-16645       14175.229               12       488.801000
   SC-20095       14142.334                9       642.833364
   HL-15040       12873.298                6      1170.299818
   SE-20110       12209.438               11       642.602000
   CC-12370       12129.072                5      1102.642909

Summary statistics:


,total_spending,order_frequency,avg_order_value
count,793.000000,793.000000,793.000000
mean,2896.848500,6.316520,227.868165
std,2628.670117,2.550885,190.342560
min,4.833000,1.000000,2.416500
25%,1146.050000,5.000000,115.520200
50%,2256.394000,6.000000,183.924000
75%,3785.276000,8.000000,282.688947
max,25043.050000,17.000000,1751.292000


### Task 5 Notes

The row-level dataset has one row per product line in an order. The new `customer_summary` table has one row per customer, which is easier to use for customer analysis and segmentation.

For each customer, the summary table stores total spending, number of unique orders, and average sales value.


---
## Task 6: Save Cleaned Data

Save the cleaned DataFrame to a new CSV file. Never overwrite the original.


In [18]:
# Save the cleaned main DataFrame.
df.to_csv('superstore_cleaned.csv', index=False)

# Save the customer summary DataFrame.
customer_summary.to_csv('customer_summary.csv', index=False)

print('Saved files:')
print('- superstore_cleaned.csv')
print('- customer_summary.csv')


Saved files:
- superstore_cleaned.csv
- customer_summary.csv


### Task 6 Notes

The cleaned order-level data was saved to `superstore_cleaned.csv`. The customer-level summary was saved to `customer_summary.csv`.

Both files are new outputs, so the original raw dataset is not overwritten.


---
## Reflection Answers

1. We used the median instead of the mean for missing numerical values because the median is less affected by outliers. For example, a few very large sales values can pull the mean upward, but the median stays closer to a typical value.

2. The row-level DataFrame has one row for each product line in an order, so it is useful for detailed product, region, and profit analysis. The customer-level summary has one row per customer, so it is better for customer segmentation and customer behavior analysis.

3. Identical rows are usually duplicates, but not always. They might represent two real transactions that happened to have the same values, especially if the dataset does not include a unique transaction ID or timestamp.
